Initial setup for the application
- Creates databases
- creates volumes
- creates tables below and populates them with initial data
  - employees
  - robots
  - shelves
  - bins

In [0]:
%sql
create database if not exists amazon_fullfilment_bronze;
create database if not exists amazon_fullfilment_silver;
create database if not exists amazon_fullfilment_gold;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.amazon_fullfilment_bronze.landing_zone;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.amazon_fullfilment_silver.processing_zone;

In [0]:
import uuid
from pyspark.sql import functions as F
from datetime import datetime

catalog_name = "workspace"
database_name = "amazon_fullfilment_bronze"

bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
silver_volume_path ="/Volumes/workspace/amazon_fullfilment_silver/processing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

# Generate a unique ID for this specific notebook run
#current_run_uuid = str(uuid.uuid4())
# 93f7758a-f5a2-4b6f-bf40-da5b67878c02
current_run_uuid = "2"

def add_bronze_metadata(df):
    """
    Standardizes the metadata for all Bronze tables.
    """
    return df.withColumn("_ingest_timestamp", F.current_timestamp()) \
             .withColumn("_source_file", F.col("_metadata.file_path")) \
             .withColumn("_batch_id", F.lit(current_run_uuid))

In [0]:
# Insert into Bronze layer function
 
def ingest_to_bronze(table_name, schema, source_path, checkpoint_path):
    """
    Standardizes the Auto Loader ingestion for any table.
    """
    raw_stream = (spark.readStream
        .format("/Volumes/workspace/default/amazon_fullfilment/source/robot_events/part-00000-tid-5755991443596397866-3d14d1f0-a5ee-47b8-af9c-2995c64171ae-262-1-c000.csvcloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .load(source_path))
    
    # We select * and _metadata to ensure the global function works
    return (add_bronze_metadata(raw_stream.select("*", "_metadata"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog_name}.{database_name}.{table_name}"))

# Creating the Bronze layer tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.orders (
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    orderdate DATE,
    status STRING NOT NULL,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,    
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
CLUSTER BY (status, order_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.order_items (
    order_id STRING NOT NULL,
    order_item_id STRING NOT NULL,
    product_id STRING NOT NULL,
    quantity INTEGER,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP NOT NULL,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
CLUSTER BY (order_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.shelves_events (
    shelf_id STRING NOT NULL,
    robot_id STRING NOT NULL,
    max_weight_capacity DOUBLE,
    status STRING NOT NULL,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (shelf_id, status)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.inventory (
    shelf_id STRING NOT NULL,
    product_id STRING NOT NULL,
    quantity INTEGER,
    order_id STRING NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (shelf_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.robot_events (
    robot_id STRING NOT NULL,
    status STRING NOT NULL,
    max_weight_capacity DOUBLE,
    battery_level INTEGER,
    event_timestamp TIMESTAMP,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (status)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_bronze.bin_events (
    bin_id STRING NOT NULL,
    order_id STRING,
    item_count INTEGER,
    status STRING NOT NULL,
    current_weight DOUBLE,
    employee_id STRING,
    updated_timestamp TIMESTAMP NOT NULL,
    _batch_id STRING,
    _ingest_timestamp TIMESTAMP,  -- When Spark processed it
    _source_file STRING           -- Which CSV it came from (for debugging)
)
USING DELTA
CLUSTER BY (bin_id)
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

# Creating Silver layer tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.customer_silver (
  customer_id STRING NOT NULL,
  first_name STRING,
  last_name STRING,
  gender STRING,                
  email STRING,
  phone STRING,
  updated_timestamp TIMESTAMP
) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE workspace.amazon_fullfilment_silver.customer_silver ALTER COLUMN customer_id SET NOT NULL;
Alter table workspace.amazon_fullfilment_silver.customer_silver  add CONSTRAINT customers_pk PRIMARY KEY(customer_id);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.address_silver (
  Address_ID STRING NOT NULL,
  Customer_ID STRING,
  Street_Address STRING,
  City STRING NOT NULL,      
  Province STRING NOT NULL,  
  Postal_Code STRING,
  Updated_Timestamp TIMESTAMP
) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.employee (
  employee_sk STRING NOT NULL,         -- Surrogate Key (usually a hash of id + start_date)
  employee_id STRING NOT NULL,
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  job_role STRING NOT NULL,      
  certification_level STRING,
  current_station_id STRING,
  shift_id STRING,
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)
USING DELTA
CLUSTER BY (employee_id)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.columnMapping.mode = 'name'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.products_silver (
  product_id STRING NOT NULL,
  sku STRING,
  product_name STRING NOT NULL,
  category STRING NOT NULL,
  weight_kg DOUBLE,
  is_current BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
 Updated_Timestamp TIMESTAMP)
 USING DELTA
 TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.orders_silver (
  order_sk STRING NOT NULL,         -- Surrogate Key (usually a hash of id + start_date)
  order_id STRING NOT NULL,         -- Natural Key
  customer_id STRING NOT NULL,
  orderdate DATE,
  status STRING,
  updated_timestamp TIMESTAMP,
  is_current BOOLEAN NOT NULL,      -- Flag for the latest record
  start_date TIMESTAMP NOT NULL,    -- Beginning of this status version
  end_date TIMESTAMP,               -- Null if current, or timestamp of status change
  
  -- Constraints for Modeling Excellence
  CONSTRAINT orders_pk PRIMARY KEY(order_sk),
  CONSTRAINT orders_customer_fk FOREIGN KEY(customer_id) 
    REFERENCES workspace.amazon_fullfilment_silver.customer_silver(customer_id)
 )
 USING DELTA
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.columnMapping.mode = 'name'
 )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.order_items_silver (
  order_item_sk STRING NOT NULL,   -- Hash of order_item_id
  order_item_id STRING NOT NULL,   -- Natural Key from source
  order_id STRING NOT NULL,         -- FK to Orders
  product_id STRING NOT NULL,       -- FK to Products
  quantity INTEGER,
  unit_price DECIMAL(10,2),        -- Good practice to include price at time of order
  updated_timestamp TIMESTAMP,
  
  -- Constraints
  CONSTRAINT order_items_pk PRIMARY KEY(order_item_sk)
 )
 USING DELTA
 CLUSTER BY (order_id, product_id) 
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true
 );

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.inventory_silver (
  inventory_sk STRING NOT NULL,   -- Hash of inventory_id
  shelf_id STRING NOT NULL,   -- Natural Key from source
  product_id STRING NOT NULL,       -- FK to Products
  quantity INTEGER,
  order_id STRING,
  updated_timestamp TIMESTAMP,
  
  -- Constraints
  CONSTRAINT inventory_pk PRIMARY KEY(inventory_sk)
 )
 USING DELTA
 CLUSTER BY (shelf_id, product_id) 
 TBLPROPERTIES (
   delta.enableChangeDataFeed = true
 );

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.robots (
  robot_sk STRING NOT NULL,        -- Surrogate Key
  robot_id STRING NOT NULL,        -- Primary Key
  robot_type STRING,               -- 'Heavy_Lifter', 'Sorter', 'Picker'

  -- Hardware Specifications (Static)
  max_weight_capacity DOUBLE,      -- To prevent "Out of Balance" robot picking
  max_speed_mps DOUBLE,            -- Meters per second for efficiency logic
  
  -- Current Operational State (Dynamic)
  status STRING,                   -- 'Idle', 'Active', 'Charging', 'Maintenance', 'Error'
  current_battery_pct INT,      -- Critical for "Return to Base" logic
  last_maintenance_date DATE,
  
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)

USING DELTA
CLUSTER BY (robot_id) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.shelves (
  shelf_sk STRING NOT NULL,        -- Surrogate Key
  shelf_id STRING NOT NULL,        -- Primary Key

  -- Physical Characteristics
  robot_id STRING,         -- Link back to the robot (if being carried)
  max_weight_capacity DOUBLE,         
  status STRING,                   -- 'In_Storage', 'In_Transit', 'Being_Restocked'

  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  Updated_Timestamp TIMESTAMP)

USING DELTA
CLUSTER BY (shelf_id) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_silver.bins (
  bins_sk STRING NOT NULL,        -- Surrogate Key
  bin_id STRING NOT NULL,          -- Unique identifier (often a barcode)

  -- Order Context
  order_id STRING,         -- The specific order this bin is collecting for
  item_count INT,                  -- Number of items currently in the bin
  -- Workflow State
  status STRING,               -- 'Empty', 'Collecting', 'Full_Waiting_Pack', 'At_Packing', 'Labeled'
  current_weight DOUBLE,
  
  -- Audit
  employee_id STRING,         -- ID of the last person to touch the bin
  is_active BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  updated_timestamp TIMESTAMP
) 
USING DELTA
CLUSTER BY (bin_id)
TBLPROPERTIES (delta.enableChangeDataFeed = true);